# Files => os, sys & shutil

`os` talks to the **operating system**, `sys` to the **Python interpreter**, and `shutil` copies and moves **files and folders**.

| Tool | Purpose | Example |
|---|---|---|
| `os.getcwd()` / `os.chdir(p)` | Current folder / change it | |
| `os.listdir(p)` / `os.scandir(p)` | Names in a folder | `os.listdir(".")` |
| `os.makedirs(p, exist_ok=True)` | Create folders (with parents) | |
| `os.rename(a, b)` / `os.remove(p)` / `os.rmdir(p)` | Rename / delete a file / delete an empty folder | |
| `os.walk(p)` | Visit a whole folder tree | |
| `os.environ` / `os.getenv(name)` | Environment variables | `os.getenv("HOME")` |
| `os.path.join` / `exists` / `splitext` / `basename` | Path helpers (`pathlib` is preferred) | |
| `os.cpu_count()` / `os.name` / `os.sep` | Machine and platform facts | |
| `sys.argv` | Command-line arguments | |
| `sys.exit(code)` | Stop the program | `sys.exit(1)` |
| `sys.path` / `sys.modules` | Import search folders / loaded modules | |
| `sys.version_info` / `sys.platform` | Python version / operating system | |
| `sys.stdin` / `stdout` / `stderr` | Standard streams | |
| `sys.getsizeof(obj)` | Size of an object in bytes | |
| `shutil.copy` / `copytree` / `move` | Copy a file / a folder / move | |
| `shutil.rmtree(p)` | Delete a folder **and everything in it** | |
| `shutil.which(name)` | Find a program on the `PATH` | |

```python
import os, sys, shutil
```

---

## `os`

### Folders and Files

```python
os.makedirs("a/b/c", exist_ok=True)
os.listdir("a")            # names only, in no guaranteed order
```

* `os.mkdir` creates **one** level. `os.makedirs` creates all missing parents.
* `os.remove` deletes files. `os.rmdir` deletes only **empty** folders.
* `os.walk(top)` yields `(folder, subfolders, files)` for every folder in the tree.

### Environment Variables

```python
os.environ["NAME"] = "value"       # set (for this program and its children)
os.getenv("NAME", "default")       # read, with a default
```

`os.environ["MISSING"]` raises `KeyError`. `os.getenv("MISSING")` returns `None`.

### `os.path`

| Function | Result for `"data/report.csv"` |
|---|---|
| `os.path.basename(p)` | `report.csv` |
| `os.path.dirname(p)` | `data` |
| `os.path.splitext(p)` | `('data/report', '.csv')` |
| `os.path.join("data", "x")` | `data/x` (or `data\x` on Windows) |
| `os.path.exists(p)` | `True` or `False` |

For new code, prefer `pathlib`.

---

## `sys`

### Exiting

`sys.exit(0)` means success. Any other value or message means failure. It works by raising `SystemExit`.

### Version and Platform

```python
sys.version_info >= (3, 11)      # compare versions as tuples
sys.platform                     # 'win32', 'linux' or 'darwin'
```

### Standard Streams

* `print()` writes to `sys.stdout`.
* Errors belong on `sys.stderr`: `print("problem", file=sys.stderr)`.

---

## `shutil`

| Task | Call |
|---|---|
| Copy a file | `shutil.copy(src, dst)` |
| Copy a file with its metadata | `shutil.copy2(src, dst)` |
| Copy a folder tree | `shutil.copytree(src, dst)` |
| Move or rename | `shutil.move(src, dst)` |
| Delete a folder tree | `shutil.rmtree(path)` |
| Create a zip archive | `shutil.make_archive(base_name, "zip", folder)` |
| Free disk space | `shutil.disk_usage(path)` |

### Important

* `shutil.rmtree` is **permanent**. Check the path twice.
* `os.system()` runs a shell command. Prefer the `subprocess` module for running programs.

## Source

https://docs.python.org/3/library/os.html

https://docs.python.org/3/library/sys.html

https://docs.python.org/3/library/shutil.html

In [ ]:
import os
import shutil
import sys
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    start = os.getcwd()
    os.chdir(tmp)

    # Folders and files
    os.makedirs("project/data/raw", exist_ok=True)
    with open("project/data/notes.txt", "w") as file:
        file.write("hello")
    with open("project/data/raw/a.txt", "w") as file:
        file.write("a")
    print(sorted(os.listdir("project/data")))

    # os.walk visits the whole tree
    for folder, subfolders, files in sorted(os.walk("project")):
        print(folder.replace(os.sep, "/"), sorted(subfolders), sorted(files))

    # os.path helpers
    path = os.path.join("data", "report.csv")
    print(os.path.basename(path), os.path.dirname(path), os.path.splitext(path)[1])
    print(os.path.exists("project"), os.path.isfile("project"), os.path.isdir("project"))

    # Rename and delete
    os.rename("project/data/notes.txt", "project/data/renamed.txt")
    os.remove("project/data/renamed.txt")
    try:
        os.rmdir("project/data")                       # not empty: raises OSError
    except OSError as error:
        print(type(error).__name__, "for a folder that is not empty")

    # shutil: copy, move and delete whole trees
    shutil.copytree("project", "backup")
    print(sorted(os.listdir("backup/data")))
    shutil.copy("backup/data/raw/a.txt", "copy_of_a.txt")
    shutil.move("copy_of_a.txt", "backup/moved.txt")
    print(sorted(os.listdir("backup")))
    shutil.rmtree("backup")
    print(os.path.exists("backup"))

    archive = shutil.make_archive("project_zip", "zip", "project")
    print(os.path.basename(archive), os.path.getsize(archive) > 0)

    os.chdir(start)                                    # leave the folder before it is deleted

# Environment variables
os.environ["DEMO_NAME"] = "value"
print(os.getenv("DEMO_NAME"), os.getenv("DEMO_MISSING"), os.getenv("DEMO_MISSING", "default"))
try:
    os.environ["DEMO_MISSING"]
except KeyError as error:
    print("KeyError for a missing variable")
del os.environ["DEMO_NAME"]

# Facts about the machine and platform
print(os.cpu_count() >= 1, os.name in ("nt", "posix"), os.sep in ("/", "\\"))
print(sys.version_info >= (3, 11), sys.platform in ("win32", "linux", "darwin"))
print(shutil.which("definitely-not-a-program"), shutil.disk_usage(".").total > 0)

# sys: exit, streams and object sizes
try:
    sys.exit(3)
except SystemExit as error:
    print("SystemExit with code", error.code)

sys.stdout.write("written to stdout\n")
print("goes to stderr", file=sys.stderr)
print(sys.getsizeof([]) < sys.getsizeof(list(range(100))), "sys" in sys.modules, type(sys.path).__name__)